In [ ]:
import pandas as pd 
import numpy as np 
from sklearn.linear_model import LogisticRegression


df = pd.read_csv("history.csv")

def h2h(df: pd.DataFrame, date: pd.DatetimeIndex, HomeTeam: str, AwayTeam: str):
    # limit dates to only the games before the match, excluding it
    df = df[df["Date"] < date]
    # choose games where the teams face each other
    df = df[(df["HomeTeam"] == HomeTeam) | (df["HomeTeam"] == AwayTeam)]
    df = df[(df["AwayTeam"] == HomeTeam) | (df["AwayTeam"] == AwayTeam)]
    score = [0,0,0]
    for row in df.itertuples():
        if row.FTR == "H":
            score[0] += 1
        elif row.FTR == "D":
            score [1] += 1
        else: 
            score[2] += 1
    return score 
            
            
match = df.iloc[5980]
h2h(df, match.Date, match.HomeTeam, match.AwayTeam)

In [ ]:
df = pd.read_csv("history.csv")
recordFeat = []
for match in df.itertuples():
    record = h2h(df, match.Date, match.HomeTeam, match.AwayTeam)
    recordFeat.append(record)


In [ ]:
recordFeat[7888]
record = [2,0,1]
res = (record[0] - record[2])/ (record[0] + record[1] + record[2] + 10**-10)
res

In [ ]:
df = pd.read_csv("history.csv", low_memory=True)
if "WR" in df.columns:
    df.drop(columns=["WR","MP"], inplace=True)

wrMargin = []
matchesPlayed = []
for record in recordFeat:
    wrMargin.append((record[0] - record[2])/ (record[0] + record[1] + record[2] + 1**-5))
    matchesPlayed.append(record[0] + record[1] + record[2])
data = {"WR": wrMargin, "MP": matchesPlayed}

dfRecords = pd.DataFrame(data)

feats = pd.concat([df,dfRecords], axis=1)
feats
feats.to_csv("history.csv", index=False)

In [ ]:
from dataclasses import dataclass, field


BASE_ELO = 1500.0
HOME_ADVANTAGE = 60.0          # added to home rating for win-prob calc only
K_FACTOR = 20.0
SEASON_REGRESSION = 0.25       # fraction pulled toward league mean each season
PROMOTED_PENALTY = 100.0       # promoted teams start this far below league mean


@dataclass
class EloSystem:
    ratings: dict = field(default_factory=dict)
    seen_this_season: set = field(default_factory=set)   # teams active in the CURRENT season
    seen_last_season: set = field(default_factory=set)   # teams active in the season just concluded
    current_season: object = None
    completed_seasons: int = 0   # how many season *transitions* we've been through

    # ---------- core probability model ----------

    def expected_home_win_prob(self, home_elo: float, away_elo: float) -> float:
        """Standard Elo logistic expectation, with home advantage baked in."""
        diff = (home_elo + HOME_ADVANTAGE) - away_elo
        return 1.0 / (1.0 + 10 ** (-diff / 400))

    # ---------- rating access / initialization ----------

    def get_rating(self, team: str) -> float:
        never_seen = team not in self.ratings
        # "Returning" = has a rating on file, but didn't play last season
        # AND hasn't already been re-priced this season (guards against
        # re-triggering the reset on every subsequent match this season).
        returning_after_absence = (
            not never_seen
            and self.completed_seasons > 0
            and team not in self.seen_last_season
            and team not in self.seen_this_season
        )

        if never_seen or returning_after_absence:
            if self.completed_seasons > 0:
                # Either a genuine first-ever promotion, or a team coming
                # back up after being relegated out of the league for a
                # while. Treat both the same way: reset to a fixed
                # below-average rating, since in both cases the squad's
                # current quality is a real unknown relative to a team
                # that's been continuously playing in this league.
                league_mean = sum(self.ratings.values()) / len(self.ratings)
                self.ratings[team] = league_mean - PROMOTED_PENALTY
            else:
                # Still within the very first season we've ever processed:
                # flat baseline for everyone.
                self.ratings[team] = BASE_ELO

        return self.ratings[team]

    # ---------- season boundary handling ----------

    def maybe_start_new_season(self, season) -> None:
        """Call this before processing each match. Detects season changes
        and applies regression-to-mean -- but ONLY to teams that actually
        played last season. Teams sitting out (relegated) are left frozen;
        they get repriced as "returning" the moment they reappear, via
        get_rating(), rather than drifting toward the mean while absent."""
        if self.current_season is None:
            self.current_season = season
            return

        if season != self.current_season:
            # Mean computed only over teams that were actually part of the
            # league last season -- stale relegated teams shouldn't drag
            # on what "average" means for the teams still playing.
            active_ratings = [self.ratings[t] for t in self.seen_this_season]
            league_mean = sum(active_ratings) / len(active_ratings)

            for team in self.seen_this_season:
                self.ratings[team] = (
                    # (1 - SEASON_REGRESSION) * self.ratings[team]
                    # + SEASON_REGRESSION * league_mean
                    1500 + (self.ratings[team] - 1500) * 0.7
                )
            # Teams NOT in seen_this_season (i.e. relegated/absent) are
            # simply left untouched here.

            self.seen_last_season = self.seen_this_season
            self.current_season = season
            self.seen_this_season = set()
            self.completed_seasons += 1

    # ---------- match processing ----------

    def process_match(self, home: str, away: str, result: str, season) -> dict:
        """
        result: 'H', 'D', or 'A' (from football-data.co.uk FTR column)

        Returns the pre-match ratings and predicted probability so you can
        use them as FEATURES for that match (this is the leakage-safe part:
        we read ratings, compute probability, THEN update).
        """
        self.maybe_start_new_season(season)

        home_elo_before = self.get_rating(home)
        away_elo_before = self.get_rating(away)

        prob_home_win = self.expected_home_win_prob(home_elo_before, away_elo_before)

        # Convert result to actual score for Elo update purposes.
        # Draw counts as 0.5 for both sides, standard chess-Elo convention.
        actual_home_score = {"H": 1.0, "D": 0.5, "A": 0.0}[result]

        new_home_elo = home_elo_before + K_FACTOR * (actual_home_score - prob_home_win)
        new_away_elo = away_elo_before + K_FACTOR * ((1 - actual_home_score) - (1 - prob_home_win))

        self.ratings[home] = new_home_elo
        self.ratings[away] = new_away_elo

        self.seen_this_season.add(home)
        self.seen_this_season.add(away)

        return {
            "home_elo_pre_match": home_elo_before,
            "away_elo_pre_match": away_elo_before,
            "elo_prob_home_win": prob_home_win,
        }
    
eloSys = EloSystem()
eloFeats = []
for row in df.itertuples():
    elos = eloSys.process_match(home=row.HomeTeam, away=row.AwayTeam, result=row.FTR, season=row.season)
    eloFeats.append(elos)
featDf = pd.DataFrame(eloFeats)
data = pd.concat([df, featDf], axis=1)
data["elo_diff"] = (data["home_elo_pre_match"] - data["away_elo_pre_match"]) 
feats = pd.concat([data,dfRecords],axis=1)
feats

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss, accuracy_score
model = LogisticRegression(max_iter=1000)
le = LabelEncoder()
categories = ["H", "D", "A"]

le.fit(categories)

train = feats[(feats["season"] > 2011) & (feats["season"] < 2023)]
test = feats[feats["season"] >= 2023]

trainX = train[["WR","MP","elo_diff"]]
trainY = train["FTR"]

testX = test[["WR","MP","elo_diff"]]
testY = test["FTR"]

model = model.fit(trainX,trainY)

probs = model.predict_proba(testX)
preds = model.predict(testX)
 
model_logloss = log_loss(testY, probs, labels=model.classes_)
model_acc = accuracy_score(testY, preds)
 
naive_probs = np.tile([1/3, 1/3, 1/3], (len(testY), 1))  # coin-flip 3-way baseline
naive_logloss = log_loss(testY, naive_probs, labels=model.classes_)
 
print(f"\nLogistic regression -- log loss: {model_logloss:.4f}, accuracy: {model_acc:.3f}")
print(f"Naive uniform baseline -- log loss: {naive_logloss:.4f}")

In [ ]:
# TODO: test with elo and compare log loss